<div dir="rtl" style="text-align: center; padding: 20px; font-family: Vazir, Tahoma, sans-serif;">
  <h1 style="margin: 0; font-size: 30px;">
تکلیف سوم  </h1>
  </p>
</div>

<hr style="border-color: #334155;">



## <div dir="rtl" style="text-align: right; font-family: Vazir, Tahoma, sans-serif;">اطلاعات کلی</div>

<div dir="rtl" style="text-align: right; line-height: 2; padding: 18px; background-color: #0f172a; color: #f8fafc; border-radius: 14px; font-family: Vazir, Tahoma, sans-serif;">
این تمرین، شامل دو بخش اصلی می‌باشد که پاسخ‌دهنده، باید هر دو بخش آن‌را پیاده‌سازی نماید.
</div>

<div dir="rtl" style="text-align: right; font-family: Vazir, Tahoma, sans-serif; line-height: 2;">
<b>نام و نام خانوادگی: امیرحسین راحتی</b><br>
<b>شماره دانشجویی: 810104303</b>
</div>

<div dir="rtl" style="text-align: right; line-height: 2; padding: 18px; background-color: #0f172a; color: #f8fafc; border-radius: 14px; font-family: Vazir, Tahoma, sans-serif;">

<b>مهلت تحویل:</b> با توجه به زمان بارگذاری تمرین، اعلام خواهد شد.<br>
<b>شیوه‌ی تحویل:</b> ارسال فایل کامل همین نوت‌بوک.<br>
<b>نحوه‌ی نمره‌دهی:</b> تمرین به صورت کلی ۱۰۰ نمره دارد که به صورت مساوی، به هر یک از دو بخش اصلی تمرین تقسیم می‌شود. ملاک نمره‌دهی، پیاه‌سازی‌های صحیح و کارآمد خواهد بود. <br>
<b>نسخه‌ی پیشنهادی پایتون:</b> Python 3.11

<hr style="border-color: #334155;">

<b>اجرای محلی (Local):</b><br>
برای اجرای این تکلیف روی سیستم شخصی، فقط این موارد را لازم دارید:
<ul style="margin-top: 8px;">
  <li>Python 3.11</li>
  <li>Jupyter Notebook یا JupyterLab</li>
  <li>فایل نوت‌بوک</li>
</ul>




</div>

<div dir="rtl" style="text-align: center; padding: 20px; font-family: Vazir, Tahoma, sans-serif;">
  <h1 style="margin: 0; font-size: 30px;">
    تکلیف ۳ (بخش اول): پیاده‌سازی Autoregressive Text Generation Loop
  </h1>
  <p style="margin-top: 12px; font-size: 16px; color: #555;">
    Autoregressive Decoding • Temperature Control • Perplexity Audit (Part 1)
  </p>
</div>

## Section Overview & Objective

In lectures, you learned that causal Large Language Models generate text by processing an input context, outputting real-valued scores called **logits** ($u$), passing them through a **softmax** function to yield a token probability distribution ($y$), and repeatedly sampling the next token.

In this assignment, you will build the complete **autoregressive generation pipeline**. You are provided with a simulated Transformer block output class that generates a logit vector of shape $[1 \times |V|]$ given an input sequence. Your task is to implement the mathematical logic for vocabulary mapping, greedy decoding, random sampling, and temperature-scaled distribution reshaping.

## Part 1: Vocabulary Mapping & Base Softmax (10 Marks)

### Task 1.1: Standard Softmax Activation

Implement a function that takes a raw, real-valued score vector $u$ (logits) of length $|V|$ and converts it into a stable probability distribution $y$ using the standard softmax formula:

$$y_i = \frac{\exp(u_i)}{\sum_{j=1}^{|V|} \exp(u_j)}$$

**TIP:** To protect against floating-point overflow from exponential operations, subtract the maximum value in the logits list from every element before exponentiating.

### Task 1.2: Cross-Entropy Loss Calculator

During pretraining, a language model utilizes Teacher Forcing to evaluate its prediction against the actual true target next token. Implement a function that calculates the single-token Cross-Entropy Loss ($L_{CE}$) given your predicted softmax distribution vector $\hat{y}$ and the integer vocabulary index of the true target token $w_{t+1}$:

$$L_{CE} = -\ln (\hat{y}[w_{t+1}])$$

In [20]:
import math
import random
import json

def standard_softmax(logits: list[float]) -> list[float]:
    """
    Computes the standard softmax probability distribution over a list of raw logits.
    
    TIP: To protect against floating-point overflow from exponential operations,
    subtract the maximum value in the logits list from every element before exponentiating.
    """
    # ==========================================
    # Your answer BEGINS HERE
    # ==========================================
    
    max_logit = max(logits)
    stable_logits = [x - max_logit for x in logits]
    exp_vals = [math.exp(x) for x in stable_logits]
    sum_exp = sum(exp_vals)
    probabilities = [e / sum_exp for e in exp_vals]
    return probabilities

    # ==========================================
    # Your answer ENDS HERE
    # ==========================================


def calculate_cross_entropy(probabilities: list[float], target_token_idx: int) -> float:
    """
    Returns the negative log probability assigned to the true target token.
    """
    # ==========================================
    # Your answer BEGINS HERE
    # ==========================================
    
    prob_target = probabilities[target_token_idx]
    log_prob = math.log(prob_target)
    ce_loss = -log_prob
    return ce_loss

    # ==========================================
    # Your answer ENDS HERE
    # ==========================================

**Part 1 Validation Code:**

In [21]:
# test standard softmax and numerical stability
test_logits = [1.2, 0.9, 0.1, -0.5]
try:
    probs = standard_softmax(test_logits)
    if probs is not None:
        print("Softmax Probabilities:", [round(p, 4) for p in probs])
        assert abs(sum(probs) - 1.0) < 1e-6, "Probabilities must sum to 1"
except Exception as e:
    print("standard_softmax implementation is not finished or raised an error:", e)
    probs = None

# test stability against large values (should run without overflow)
overflow_logits = [1000.0, 999.0, 990.0]
try:
    overflow_probs = standard_softmax(overflow_logits)
    if overflow_probs is not None:
        print("Overflow-safe Probabilities:", [round(p, 4) for p in overflow_probs])
except OverflowError:
    print("Error: standard_softmax is not overflow-safe!")
except Exception as e:
    pass

# test cross entropy loss
if probs is not None:
    test_target_idx = 1 # target is "the" (index 1)
    try:
        ce_loss = calculate_cross_entropy(probs, test_target_idx)
        print(f"Cross Entropy Loss for target index {test_target_idx}: {ce_loss:.4f}")
        assert ce_loss > 0, "Loss must be positive"
    except Exception as e:
        print("calculate_cross_entropy implementation is not finished or raised an error:", e)

Softmax Probabilities: [0.4432, 0.3283, 0.1475, 0.081]
Overflow-safe Probabilities: [0.731, 0.2689, 0.0]
Cross Entropy Loss for target index 1: 1.1138


## Part 2: Implementing Decoding Strategies (14 Marks)

### Task 2.1: Greedy Decoding

Implement **Greedy Decoding**, which isolates and selects the locally optimal token with the absolute maximum likelihood probability at the current time step:

$$\vec{w}_t = \arg\max_{w \in V} P(w \mid w_{<t})$$

### Task 2.2: Random (Multinomial) Sampling

Implement a categorical random selector. Instead of selecting the absolute highest value, your code must select a random vocabulary index *according to the specific probability weights* defined by the distribution array.

In [22]:
def decode_greedy(probabilities: list[float]) -> int:
    """
    Returns the integer vocabulary index of the token with the highest probability.
    """
    # ==========================================
    # Your answer BEGINS HERE
    # ==========================================
    
    max_prob = max(probabilities)
    best_index = probabilities.index(max_prob)
    return best_index

    # ==========================================
    # Your answer ENDS HERE
    # ==========================================


def decode_random_sampling(probabilities: list[float]) -> int:
    """
    Randomly selects and returns a token index based on the probability weights 
    using a random float interval roll in the range [0.0, 1.0).
    Do NOT use numpy or external multinomial packages.
    """
    # ==========================================
    # Your answer BEGINS HERE
    # ==========================================
    
    r = random.random()
    cumulative = 0.0
    for idx, prob in enumerate(probabilities):
        cumulative += prob
        if r < cumulative:
            return idx
    return len(probabilities) - 1

    # ==========================================
    # Your answer ENDS HERE
    # ==========================================

**Part 2 Validation Code:**

In [23]:
random.seed(42)
test_probs = [0.1, 0.6, 0.2, 0.1]

try:
    greedy_choice = decode_greedy(test_probs)
    print("Greedy choice (expected 1):", greedy_choice)
    assert greedy_choice == 1, "Greedy choice failed!"
except Exception as e:
    print("decode_greedy implementation is not finished or raised an error:", e)

try:
    # test random sampling counts over many iterations
    sim_counts = {0: 0, 1: 0, 2: 0, 3: 0}
    for _ in range(10000):
        choice = decode_random_sampling(test_probs)
        sim_counts[choice] += 1
        
    print("Simulation percentages (expected close to 10%, 60%, 20%, 10%):")
    for k, v in sim_counts.items():
        print(f"  Token {k}: {v / 10000 * 100:.2f}%")
        
    assert sim_counts[1] > sim_counts[2] > sim_counts[0], "Probability weighting is incorrect!"
except Exception as e:
    print("decode_random_sampling implementation is not finished or raised an error:", e)

Greedy choice (expected 1): 1
Simulation percentages (expected close to 10%, 60%, 20%, 10%):
  Token 0: 9.86%
  Token 1: 60.37%
  Token 2: 19.97%
  Token 3: 9.80%


## Part 3: Temperature Scaling & Reshaping (16 Marks)

### Task 3.1: Softmax with Temperature

To balance generation quality and diversity, systems inject a temperature parameter $\tau$. Write a function that divides the incoming raw logit vector array by a temperature value $\tau$ prior to compiling the softmax math:

$$y_i = \frac{\exp(u_i / \tau)}{\sum_{j=1}^{|V|} \exp(u_j / \tau)}$$

### Task 3.2: The Autoregressive Generation Loop

Put everything together. You are provided with a helper function, `get_model_logits(context_tokens: list[str]) -> list[float]`, which acts as your black-box neural decoder network layer and outputs a raw logit score array.

Implement the continuous loop that tracks output strings, transforms logit vectors into choices, appends selections back to the evolving text context buffer, and terminates immediately when encountering a special End-of-Sentence string label token (`"<EOS>"` or `"</s>"`).

In [24]:
# Vocabulary mapping for autoregressive loop testing
VOCAB_TO_IDX = {"all": 0, "the": 1, "your": 2, "that": 3, "</s>": 4}
IDX_TO_VOCAB = {0: "all", 1: "the", 2: "your", 3: "that", 4: "</s>"}

def token_split(prompt: str) -> list[str]:
    """
    A simple helper that splits text on whitespace and filters out empty values.
    """
    return [token for token in prompt.strip().split(" ") if token]

def get_model_logits(context_tokens: list[str]) -> list[float]:
    """
    A mock Causal LM that returns a raw logit vector over the vocabulary of size 5
    depending on the active trailing context.
    """
    context = [t.lower() for t in context_tokens]
    if not context:
        return [1.5, 0.5, -0.5, -1.0, -5.0]
    
    last_word = context[-1]
    if last_word == "for":
        return [1.2, 0.9, 0.1, -0.5, -10.0]
    elif last_word == "all":
        return [-10.0, -10.0, -10.0, -10.0, 5.0]  # Strong target on </s>
    else:
        return [0.1, 0.2, 0.3, 0.4, -2.0]


def softmax_with_temperature(logits: list[float], temperature: float) -> list[float]:
    """
    Reshapes the distribution based on temperature tau.
    - If temperature == 1.0, this behaves identically to standard softmax.
    - If temperature is very low (e.g., 0.1), the highest value dominates (towards greedy).
    - If temperature is very high (e.g., 10.0), values flatten out (towards uniform).
    """
    # ==========================================
    # Your answer BEGINS HERE
    # ==========================================
    
    if temperature <= 0:
        temperature = 1e-8
        
    scaled_logits = [x / temperature for x in logits]
    max_logit = max(scaled_logits)
    exp_vals = [math.exp(x - max_logit) for x in scaled_logits]
    sum_exp = sum(exp_vals)
    probabilities = [e / sum_exp for e in exp_vals]
    return probabilities

    # ==========================================
    # Your answer ENDS HERE
    # ==========================================


def generate_text(prompt: str, max_tokens: int, strategy: str, temperature: float = 1.0) -> str:
    """
    Generates text autoregressively starting from an input prompt.
    
    Parameters:
      - prompt: The initial string text supplied by the user.
      - max_tokens: Hard stop ceiling condition for loop control.
      - strategy: A string literal flag; either "greedy" or "sample".
      - temperature: The value used to scale logits if strategy is "sample".
    """
    context = token_split(prompt)
    generated_sequence = []
    
    # ==========================================
    # Your answer BEGINS HERE
    # ==========================================
    
    for _ in range(max_tokens):
        logits = get_model_logits(context)
        if strategy == "greedy":
            probabilities = softmax_with_temperature(logits, 1.0)
            next_token_idx = decode_greedy(probabilities)
        elif strategy == "sample":
            probabilities = softmax_with_temperature(logits, temperature)
            next_token_idx = decode_random_sampling(probabilities)

        next_token = IDX_TO_VOCAB[next_token_idx]
        if next_token == "</s>":
            break
        context.append(next_token)
        generated_sequence.append(next_token)
    
    # ==========================================
    # Your answer ENDS HERE
    # ==========================================
    
    return " ".join(generated_sequence)

**Part 3 Validation Code:**

In [25]:
slide_logits = [1.2, 0.9, 0.1, -0.5]

try:
    # 1. Test tau = 1.0 (should match standard softmax)
    p_1 = softmax_with_temperature(slide_logits, 1.0)
    if p_1 is not None:
        print("Tau = 1.0 (Expected: [0.4432, 0.3283, 0.1475, 0.081]):")
        print(" ", [round(p, 4) for p in p_1])
        assert abs(p_1[0] - 0.4432) < 1e-3
    
    # 2. Test very low tau
    p_low = softmax_with_temperature(slide_logits, 0.1)
    if p_low is not None:
        print("Tau = 0.1 (Expected to concentrate heavily on index 0):")
        print(" ", [round(p, 4) for p in p_low])
        assert p_low[0] > 0.95
    
    # 3. Test very high tau
    p_high = softmax_with_temperature(slide_logits, 100.0)
    if p_high is not None:
        print("Tau = 100.0 (Expected close to uniform [0.25, 0.25, 0.25, 0.25]):")
        print(" ", [round(p, 4) for p in p_high])
        assert abs(p_high[0] - 0.25) < 1e-2
except Exception as e:
    print("softmax_with_temperature implementation error:", e)

try:
    # Test autoregressive generation
    greedy_gen = generate_text("So long and thanks for", max_tokens=10, strategy="greedy")
    print(f"\nGenerated text with greedy: '{greedy_gen}'")
    assert greedy_gen == "all", "Greedy generation should have generated 'all' and then stopped at '</s>'"
except Exception as e:
    print("generate_text implementation error:", e)

Tau = 1.0 (Expected: [0.4432, 0.3283, 0.1475, 0.081]):
  [0.4432, 0.3283, 0.1475, 0.081]
Tau = 0.1 (Expected to concentrate heavily on index 0):
  [0.9526, 0.0474, 0.0, 0.0]
Tau = 100.0 (Expected close to uniform [0.25, 0.25, 0.25, 0.25]):
  [0.2519, 0.2512, 0.2492, 0.2477]

Generated text with greedy: 'all'


## Part 4: High-Throughput Validation & Perplexity Audit Pipeline (10 Marks)

In this final section, you will write an evaluation suite to stress-test your decoding logic. You will implement a high-throughput script that ingests a corpus validation matrix, handles mathematical underflow constraints across full text generation trajectories, and generates an audit report measuring language model performance under diverse decoding states.

### Task 4.1: The Corpus Evaluation & Perplexity Matrix

Complete the implementation of the evaluation pipeline below. Your script must parse a sequential time-series logit matrix, handle teacher-forced target matching, convert scaling outcomes into a uniform log-base footprint, and compute the total geometric mean to derive the model's **Perplexity ($PP$)** on the unseen sequence.

$$\text{Perplexity}(W_{1:n}) = \sqrt[n]{\prod_{i=1}^{n} \frac{1}{P(w_i \mid w_{<i})}} = \exp\left( \frac{1}{n} \sum_{i=1}^{n} L_{CE}(w_i) \right)$$

In [26]:
def run_production_audit_pipeline():
    print("==========================================================================")
    print("    COGNITIVE SYSTEMS LABORATORY - DISCRETE LLM AUDIT & VALIDATION SUITE ")
    print("==========================================================================\n")
    
    # 1. System Vocabulary Space (Size V = 6)
    vocab = {0: "the", 1: "chicken", 2: "crossed", 3: "road", 4: "it", 5: "</s>"}
    
    # 2. Evaluation Validation Dataset
    # High-throughput matrix capturing a 3-sentence batch execution trajectory.
    evaluation_corpus = [
        {
            "sequence_name": "Sentence_1_Coherent",
            "logits_trajectory": [
                [3.5, -1.0, -0.5, -2.0,  0.1, -4.0],  # Step 1: Target is "the" (0)
                [-0.5, 4.2, -1.2, -0.1,  0.8, -5.0],  # Step 2: Target is "chicken" (1)
                [-2.0, -1.0, 5.0,  1.1, -0.5, -3.0],  # Step 3: Target is "crossed" (2)
                [0.2, -1.5,  0.8, 3.9, -0.2, -1.0],  # Step 4: Target is "road" (3)
                [-3.0, -4.0, -2.5, -2.0, -1.5, 6.2]   # Step 5: Target is "</s>" (5)
            ],
            "ground_truth_indices": [0, 1, 2, 3, 5]
        },
        {
            "sequence_name": "Sentence_2_Ambiguous",
            "logits_trajectory": [
                [2.8, -0.5, -0.2, -1.5,  1.5, -3.0],  # Step 1: Target is "the" (0)
                [-0.1, 2.1, -0.5, -0.2,  2.9, -4.0],  # Step 2: Target is "it" (4)
                [-1.0, 3.2,  1.5,  0.5, -0.1, -2.5],  # Step 3: Target is "chicken" (1)
                [-2.5, -2.0, -1.5, -1.0, -0.5, 5.8]   # Step 4: Target is "</s>" (5)
            ],
            "ground_truth_indices": [0, 4, 1, 5]
        },
        {
            "sequence_name": "Sentence_3_Word_Salad_Anomaly",
            "logits_trajectory": [
                [-1.5, 0.5, 4.1, -2.0,  0.8, -3.5],  # Step 1: Target is "chicken" (1)
                [3.0, -1.2, 0.1, -0.5,  0.2, -4.0],  # Step 2: Target is "crossed" (2)
                [-0.2, 0.4, -1.0,  0.1,  1.1, 4.5]   # Step 3: Target is "the" (0)
            ],
            "ground_truth_indices": [1, 2, 0]
        }
    ]

    # ==========================================================================
    # METRIC SUB-ROUTINE 1: Perplexity Performance Benchmarking
    # ==========================================================================
    print("--- METRIC BENCHMARK 1: Computing Perplexity (PP) Across Corpus Streams ---")
    
    total_corpus_loss = 0.0
    total_corpus_tokens = 0
    
    # ==========================================
    # Your answer BEGINS HERE
    # ==========================================
    for sentence_data in evaluation_corpus:
        sentence_name = sentence_data["sequence_name"]
        logits_trajectory = sentence_data["logits_trajectory"]
        ground_truth = sentence_data["ground_truth_indices"]
        
        sentence_loss = 0.0
        num_tokens = len(ground_truth)
        
        # پردازش هر step در جمله
        for step_idx, (logits, target_idx) in enumerate(zip(logits_trajectory, ground_truth)):
            probs = standard_softmax(logits)
            ce_loss = calculate_cross_entropy(probs, target_idx)
            sentence_loss += ce_loss
            print(f"  {sentence_name} - Step {step_idx+1}: "
                  f"target='{vocab[target_idx]}', CE_loss={ce_loss:.4f}")
        
        avg_loss = sentence_loss / num_tokens
        sentence_perplexity = math.exp(avg_loss)
        
        print(f"  >>> {sentence_name} - Avg Loss: {avg_loss:.4f}, "
              f"Perplexity: {sentence_perplexity:.4f}\n")
        
        total_corpus_loss += sentence_loss
        total_corpus_tokens += num_tokens
    
    corpus_avg_loss = total_corpus_loss / total_corpus_tokens
    corpus_perplexity = math.exp(corpus_avg_loss)

    # ==========================================
    # Your answer ENDS HERE
    # ==========================================


    # ==========================================================================
    # METRIC SUB-ROUTINE 2: Dynamic Entropy and Temperature Variance Analysis
    # ==========================================================================
    print("\n--- METRIC BENCHMARK 2: Empirical Vocabulary Generation & Distribution Entropy Analysis ---")
    
    # Isolate highly volatile step token frame (Sentence 2, Step 2 logits)
    volatile_logits = evaluation_corpus[1]["logits_trajectory"][1] # [-0.1, 2.1, -0.5, -0.2, 2.9, -4.0]
    temperatures_regime = [0.1, 0.5, 1.0, 5.0]
    
    # ==========================================
    # Your answer BEGINS HERE
    # ==========================================
    
    print(f"volatile logits vector: {volatile_logits}")
    print(f"vocab: {vocab}\n")
    
    for temp in temperatures_regime:
        probs = softmax_with_temperature(volatile_logits, temp)
        entropy = 0.0
        for p in probs:
            if p > 0:
                entropy -= p * math.log(p)
        
        most_likely_idx = decode_greedy(probs)
        most_likely_token = vocab[most_likely_idx]
        
        print(f"temperature = {temp:.1f}:")
        print(f" most likely token: '{most_likely_token}' (idx={most_likely_idx})")
        print(f"entropy: {entropy:.4f}")
        print(f"prob distribution :")
        
        probs_with_idx = list(enumerate(probs))
        probs_with_idx.sort(key=lambda x: x[1], reverse=True)
        for i in range(min(3, len(probs_with_idx))):
            idx, p = probs_with_idx[i]
            print(f"{vocab[idx]}: {p:.4f}")
        print()
  
    # ==========================================
    # Your answer ENDS HERE
    # ==========================================

In [27]:
try:
    run_production_audit_pipeline()
except Exception as e:
    print("Pipeline execution failed (likely due to unfinished TODO sections):", e)

    COGNITIVE SYSTEMS LABORATORY - DISCRETE LLM AUDIT & VALIDATION SUITE 

--- METRIC BENCHMARK 1: Computing Perplexity (PP) Across Corpus Streams ---
  Sentence_1_Coherent - Step 1: target='the', CE_loss=0.0653
  Sentence_1_Coherent - Step 2: target='chicken', CE_loss=0.0589
  Sentence_1_Coherent - Step 3: target='crossed', CE_loss=0.0277
  Sentence_1_Coherent - Step 4: target='road', CE_loss=0.0938
  Sentence_1_Coherent - Step 5: target='</s>', CE_loss=0.0010
  >>> Sentence_1_Coherent - Avg Loss: 0.0493, Perplexity: 1.0506

  Sentence_2_Ambiguous - Step 1: target='the', CE_loss=0.3190
  Sentence_2_Ambiguous - Step 2: target='it', CE_loss=0.4565
  Sentence_2_Ambiguous - Step 3: target='chicken', CE_loss=0.2663
  Sentence_2_Ambiguous - Step 4: target='</s>', CE_loss=0.0043
  >>> Sentence_2_Ambiguous - Avg Loss: 0.2615, Perplexity: 1.2989

  Sentence_3_Word_Salad_Anomaly - Step 1: target='chicken', CE_loss=3.6683
  Sentence_3_Word_Salad_Anomaly - Step 2: target='crossed', CE_loss=3.0501

<div dir="rtl" style="text-align: center; padding: 20px; font-family: Vazir, Tahoma, sans-serif;">
  <h1 style="margin: 0; font-size: 30px;">
    تکلیف ۳ (بخش دوم): طراحی و پیاده‌سازی مکانیزم Multihead Attention
  </h1>
  <p style="margin-top: 12px; font-size: 16px; color: #555;">
    Architecting Multi-Head Causal Attention from Scratch (Part 2)
  </p>
</div>

## Section Overview & Objective

In lecture, you learned that instead of processing elements sequentially, the Transformer packages a sequence of $N$ tokens into a single sequence matrix $\mathbf{X}$ of shape $[N \times d]$ to execute self-attention in parallel.

In this assignment, you will implement a modular, high-throughput **Multi-Head Causal Self-Attention Layer**.

## Part 1: Linear Token Projections & Head Slicing (15 Marks)

### Task 1.1: Projecting to Q, K, V Spaces

Implement a routine that takes a batch input matrix $\mathbf{X}$ of shape $[N \times d]$ and projects it into separate Query, Key, and Value matrices ($\mathbf{Q}, \mathbf{K}, \mathbf{V}$) using learned projection weights:

* $\mathbf{Q} = \mathbf{X}W^Q$
* $\mathbf{K} = \mathbf{X}W^K$
* $\mathbf{V} = \mathbf{X}W^V$

### Task 1.2: Multi-Head Splitting

To calculate Multi-Head Attention, we divide our large sequence space into $h$ separate structural heads so that each head can attend to different conceptual vectors. Write a function that takes a matrix of shape $[N \times d]$ and splits it horizontally (across columns) into $h$ sub-matrices, each of shape $[N \times d_{head}]$, where $d_{head} = d / h$.

In [28]:
def project_inputs(X: list[list[float]], W: list[list[float]]) -> list[list[float]]:
    """
    Multiplies input matrix X [N x d] by weight matrix W [d x d_k]
    to output a projected representation matrix of shape [N x d_k].
    
    Matrix multiplication: result[i][j] = sum_k X[i][k] * W[k][j]
    """
    # ==========================================
    # Your answer BEGINS HERE
    # ==========================================
    N = len(X)
    d = len(X[0])
    d_k = len(W[0])
    
    result = [[0.0 for _ in range(d_k)] for _ in range(N)]
    
    for i in range(N):
        for j in range(d_k):
            total = 0.0
            for k in range(d):
                total += X[i][k] * W[k][j]
            result[i][j] = total
    
    return result
    # ==========================================
    # Your answer ENDS HERE
    # ==========================================


def split_heads(matrix: list[list[float]], num_heads: int) -> list[list[list[float]]]:
    """
    Splits an [N x d] matrix into a list of length 'num_heads',
    where each element is an [N x d_head] sub-matrix (d_head = d / num_heads).
    
    Example: If matrix = [N x 4] and num_heads = 2, d_head = 2
    Head 0 gets columns 0-1, Head 1 gets columns 2-3
    """
    # ==========================================
    # Your answer BEGINS HERE
    # ==========================================
    N = len(matrix)
    d = len(matrix[0])
    
    d_head = d // num_heads    
    
    heads = []
    
    for h in range(num_heads):
        start_col = h * d_head
        end_col = (h + 1) * d_head
        head_matrix = []
        for i in range(N):
            row_slice = matrix[i][start_col:end_col]
            head_matrix.append(row_slice)

        heads.append(head_matrix)
    
    return heads
    # ==========================================
    # Your answer ENDS HERE
    # ==========================================

**Part 1 Validation Code:**

In [29]:
# Define a test input matrix [3 x 4] and weight matrix [4 x 2]
X_test = [
    [1.0, 2.0, 0.0, -1.0],
    [0.0, 1.0, 3.0,  2.0],
    [-2.0, 0.0, 1.0, 1.0]
]
W_test = [
    [0.5,  1.5],
    [2.0, -0.5],
    [0.0,  1.0],
    [-1.0, 0.0]
]

try:
    proj_out = project_inputs(X_test, W_test)
    print("Projected outputs [3 x 2]:")
    for r in proj_out:
        print(" ", [round(v, 4) for v in r])
    # Validate matrix multiplication correctness
    # Row 1: [1*0.5 + 2*2 + 0 - 1*-1, 1*1.5 + 2*-0.5 + 0 + 0] = [5.5, 0.5]
    assert abs(proj_out[0][0] - 5.5) < 1e-5
    assert abs(proj_out[0][1] - 0.5) < 1e-5
    print("SUCCESS: project_inputs passes verification tests!")
except Exception as e:
    print("project_inputs implementation failed or incomplete:", e)

try:
    # Split input matrix X_test [3 x 4] into 2 heads (each [3 x 2])
    splitted = split_heads(X_test, num_heads=2)
    print("Splitted Heads:")
    for idx, head in enumerate(splitted):
        print(f"  Head {idx}:")
        for r in head:
            print("   ", r)
    assert len(splitted) == 2
    assert len(splitted[0]) == 3
    assert len(splitted[0][0]) == 2
    assert splitted[0][0] == [1.0, 2.0]
    assert splitted[1][0] == [0.0, -1.0]
    print("SUCCESS: split_heads passes verification tests!")
except Exception as e:
    print("split_heads implementation failed or incomplete:", e)

Projected outputs [3 x 2]:
  [5.5, 0.5]
  [0.0, 2.5]
  [-2.0, -2.0]
SUCCESS: project_inputs passes verification tests!
Splitted Heads:
  Head 0:
    [1.0, 2.0]
    [0.0, 1.0]
    [-2.0, 0.0]
  Head 1:
    [0.0, -1.0]
    [3.0, 2.0]
    [1.0, 1.0]
SUCCESS: split_heads passes verification tests!


## Part 2: Scaled Dot-Product & Causal Masking (20 Marks)

### Task 2.1: Causal Masking at Matrix Level

In Decoder-only models, a token at time $t$ should not be able to access information of future tokens at time $j > t$. This constraint is applied by inserting a large negative value (such as $-1e9$ serving as negative infinity $-\infty$) into the upper-triangular portion of the raw similarity score matrix.

Write a function that takes an $[N \times N]$ square matrix and replaces all elements where $j > i$ with $-1e9$.

### Task 2.2: Compute Head Attention

Compute the scaled dot-product causal self-attention for the input head matrices using the following formula:

$$\mathbf{A} = \text{softmax}\left(\text{mask}\left(\frac{\mathbf{Q}\mathbf{K}^\top}{\sqrt{d_k}}\right)\right)\mathbf{V}$$

**Implementation Steps:**
1. Compute the matrix multiplication $\mathbf{Q}\mathbf{K}^\top$ to get an $[N \times N]$ raw similarity matrix.
2. Scale the scores by dividing all elements by $\sqrt{d_k}$.
3. Apply the causal mask to the scaled similarity matrix (by calling the function from the previous task).
4. Compute a stable softmax distribution row-by-row (by subtracting the maximum value of each row before exponentiating).
5. Multiply the resulting attention probability weight matrix by $\mathbf{V}$ to get the final output representation of the head.

In [30]:
import math

def apply_causal_mask(scores: list[list[float]]) -> list[list[float]]:
    """
    Takes an [N x N] score matrix and replaces all elements where j > i 
    with a large negative value (e.g., -1e9) to serve as -infinity.
    
    For a matrix index [i][j]:
    - i = row index (position of current token / query)
    - j = column index (position of key / token being attended to)
    - Condition j > i means attending to FUTURE tokens -> mask out
    """
    # ==========================================
    # Your answer BEGINS HERE
    # ==========================================
    N = len(scores)
    
    masked_scores = [row[:] for row in scores]
    
    for i in range(N):
        for j in range(N):
            if j > i:
                masked_scores[i][j] = -1e9
    
    return masked_scores
    # ==========================================
    # Your answer ENDS HERE
    # ==========================================


def compute_head_attention(Q_head: list[list[float]], K_head: list[list[float]], V_head: list[list[float]], d_k: int) -> list[list[float]]:
    """
    1. Computes Q dot K^T to get a raw similarity matrix [N x N].
    2. Scales the scores dividing by math.sqrt(d_k).
    3. Applies your apply_causal_mask() function.
    4. Runs row-wise stable softmax normalization using standard_softmax().
    5. Multiplies the resulting attention weights by V_head.
    """
    # ==========================================
    # Your answer BEGINS HERE
    # ==========================================
    N = len(Q_head)
    
    K_transpose = []
    for col_idx in range(len(K_head[0])):
        new_row = []
        for row_idx in range(N):
            new_row.append(K_head[row_idx][col_idx])
        K_transpose.append(new_row)
    
    raw_scores = []
    for i in range(N):
        score_row = []
        for j in range(N):
            total = 0.0
            for k in range(d_k):
                total += Q_head[i][k] * K_transpose[k][j]
            score_row.append(total)
        raw_scores.append(score_row)
    
    scale_factor = math.sqrt(d_k)
    scaled_scores = [[val / scale_factor for val in row] for row in raw_scores]
    
    masked_scores = apply_causal_mask(scaled_scores)
    
    attention_weights = []
    for row in masked_scores:
        calculated_softmax = standard_softmax(row)  
        attention_weights.append(calculated_softmax)
    

    d_head = len(V_head[0])
    output = []
    for i in range(N):
        output_row = []
        for j in range(d_head):
            total = 0.0
            for k in range(N):
                total += attention_weights[i][k] * V_head[k][j]
            output_row.append(total)
        output.append(output_row)
    
    return output
    # ==========================================
    # Your answer ENDS HERE
    # ==========================================

**Part 2 Validation Code:**

In [31]:
test_scores = [
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0],
    [7.0, 8.0, 9.0]
]

try:
    masked_scores = apply_causal_mask(test_scores)
    print("Masked score matrix:")
    for r in masked_scores:
        print(" ", r)
    assert masked_scores[0][1] == -1e9
    assert masked_scores[0][2] == -1e9
    assert masked_scores[1][2] == -1e9
    assert masked_scores[1][0] == 4.0
    print("SUCCESS: apply_causal_mask passes verification tests!")
except Exception as e:
    print("apply_causal_mask implementation failed or incomplete:", e)

# Test per-head attention calculation
Q_h = [[1.0, 0.0], [0.0, 1.0]]
K_h = [[1.0, 0.0], [0.0, 1.0]]
V_h = [[10.0, 20.0], [30.0, 40.0]]

try:
    head_out = compute_head_attention(Q_h, K_h, V_h, d_k=2)
    print("Head self-attention output [2 x 2]:")
    for r in head_out:
        print(" ", r)
    # Row 0 (token 1) should only attend to itself due to causal mask -> output must equal V_h[0]
    assert abs(head_out[0][0] - 10.0) < 1e-5
    assert abs(head_out[0][1] - 20.0) < 1e-5
    print("SUCCESS: compute_head_attention passes verification tests!")
except Exception as e:
    print("compute_head_attention implementation failed or incomplete:", e)

Masked score matrix:
  [1.0, -1000000000.0, -1000000000.0]
  [4.0, 5.0, -1000000000.0]
  [7.0, 8.0, 9.0]
SUCCESS: apply_causal_mask passes verification tests!
Head self-attention output [2 x 2]:
  [10.0, 20.0]
  [23.39523098653314, 33.395230986533136]
SUCCESS: compute_head_attention passes verification tests!


## Part 3: Concatenation and Output Projection (15 Marks)

### Task 3.1: Multi-Head Attention Layer

Implement the complete Multi-Head Self-Attention block. The function `multi_head_attention_layer` should execute the following steps:
1. Project the input matrix $\mathbf{X}$ into Query, Key, and Value spaces using weights $W^Q$, $W^K$, and $W^V$ (by calling `project_inputs`).
2. Split the projected matrices into the specified number of concurrent heads (`num_heads`) to get Q, K, and V heads for each branch (by calling `split_heads`).
3. For each head $i$, compute the causal self-attention output by calling `compute_head_attention`.
4. Concatenate the output representations of the heads along the column dimension to reconstruct a single matrix of shape $[N \times d]$.
5. Multiply the concatenated matrix by the output projection weight matrix $W^O$ to produce the final layer output.

In [32]:
def multi_head_attention_layer(X: list[list[float]], config: dict) -> list[list[float]]:
    """
    Performs the complete Multi-Head Attention update mapping pass.
    
    config dictionary includes:
      - 'W_Q', 'W_K', 'W_V': Projection matrices [d x d]
      - 'W_O': Final output blend matrix [d x d]
      - 'num_heads': Number of concurrent structural heads (h)
      - 'd_k': Hidden dimension size per head (d_k = d / num_heads)
    """
    # ==========================================
    # Your answer BEGINS HERE
    # ==========================================
    
    W_Q = config['W_Q']
    W_K = config['W_K']
    W_V = config['W_V']
    W_O = config['W_O']
    num_heads = config['num_heads']
    d_k = config['d_k']
    
    Q = project_inputs(X, W_Q)
    K = project_inputs(X, W_K)
    V = project_inputs(X, W_V)
    
    Q_heads = split_heads(Q, num_heads)
    K_heads = split_heads(K, num_heads)
    V_heads = split_heads(V, num_heads)
    
    head_outputs = []
    for i in range(num_heads):
        Q_head = Q_heads[i]
        K_head = K_heads[i]
        V_head = V_heads[i]
        
        head_out = compute_head_attention(Q_head, K_head, V_head, d_k)
        head_outputs.append(head_out)

    N = len(X) 
    concatenated = []
    for i in range(N):
        row = []
        for head_mat in head_outputs:
            row.extend(head_mat[i])
        concatenated.append(row)
    
    output = project_inputs(concatenated, W_O)
    
    return output
    # ==========================================
    # Your answer ENDS HERE
    # ==========================================

**Part 3 Validation Code:**

In [33]:
# Mock layer with Identity weights to test aggregation correctness
config_test = {
    "num_heads": 2,
    "d_k": 2,
    "W_Q": [[1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0], [0.0, 0.0, 1.0, 0.0], [0.0, 0.0, 0.0, 1.0]],
    "W_K": [[1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0], [0.0, 0.0, 1.0, 0.0], [0.0, 0.0, 0.0, 1.0]],
    "W_V": [[1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0], [0.0, 0.0, 1.0, 0.0], [0.0, 0.0, 0.0, 1.0]],
    "W_O": [[1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0], [0.0, 0.0, 1.0, 0.0], [0.0, 0.0, 0.0, 1.0]]
}

try:
    out_layer = multi_head_attention_layer(X_test, config_test)
    print("Multi-Head Layer Output [3 x 4]:")
    for r in out_layer:
        print(" ", [round(v, 4) for v in r])
    
    # Row 0 only attends to itself due to causal mask -> output must equal input
    assert abs(out_layer[0][0] - X_test[0][0]) < 1e-5
    assert abs(out_layer[0][3] - X_test[0][3]) < 1e-5
    print("SUCCESS: multi_head_attention_layer passes verification tests!")
except Exception as e:
    print("multi_head_attention_layer implementation failed or raised an error:", e)

Multi-Head Layer Output [3 x 4]:
  [1.0, 2.0, 0.0, -1.0]
  [0.6698, 1.6698, 2.9999, 1.9999]
  [-1.8497, 0.0818, 2.7506, 1.8563]
SUCCESS: multi_head_attention_layer passes verification tests!


## Part 4: Practical Execution & Pipeline Verification

In this section, you will evaluate the behavior of your multi-head causal attention mechanism on a sample sentence:
`["The", "chicken", "road", "it"]`

In this audit, the fourth row corresponds to the word `"it"` (an ambiguous pronoun). Verify that, due to the causal mask, the word `"it"` can only attend to itself and preceding tokens (indices 0 to 3).

Run and observe the output of this validation pipeline.

In [34]:
def run_transformer_validation_pipeline():
    print("=== STARTING MULTI-HEAD ATTENTION EMBEDDING AUDIT PIPELINE ===\n")
    
    # 4 tokens: ["The", "chicken", "road", "it"] | Embedding dimension d = 4
    X = [
        [0.25,  0.10, -0.15,  0.05],  # "The"
        [0.85,  0.90,  0.05,  0.40],  # "chicken"
        [-0.10, 0.30,  0.95,  0.10],  # "road"
        [0.50,  0.45,  0.20,  0.35]   # "it"
    ]
    
    config = {
        "num_heads": 2,
        "d_k": 2,
        "W_Q": [[1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0], [0.0, 0.0, 1.0, 0.0], [0.0, 0.0, 0.0, 1.0]],
        "W_K": [[1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0], [0.0, 0.0, 1.0, 0.0], [0.0, 0.0, 0.0, 1.0]],
        "W_V": [[1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0], [0.0, 0.0, 1.0, 0.0], [0.0, 0.0, 0.0, 1.0]],
        "W_O": [[1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0], [0.0, 0.0, 1.0, 0.0], [0.0, 0.0, 0.0, 1.0]]
    }
    
    try:
        output_embeddings = multi_head_attention_layer(X, config)
        print("Transformed Embeddings Output:")
        for idx, token_vec in enumerate(output_embeddings):
            print(f"  Token {idx}: {[round(v, 4) for v in token_vec]}")
        print("\nVerification assertion: check causal masking limits.")
        print("Layer context successfully transformed and verified.")
    except Exception as e:
        print("Pipeline execution failed (likely due to unfinished TODO sections):", e)

if __name__ == "__main__":
    run_transformer_validation_pipeline()

=== STARTING MULTI-HEAD ATTENTION EMBEDDING AUDIT PIPELINE ===

Transformed Embeddings Output:
  Token 0: [0.25, 0.1, -0.15, 0.05]
  Token 1: [0.6728, 0.6638, -0.0447, 0.2343]
  Token 2: [0.3458, 0.4502, 0.4457, 0.1706]
  Token 3: [0.446, 0.4976, 0.2831, 0.2284]

Verification assertion: check causal masking limits.
Layer context successfully transformed and verified.
